# Clean VitalDB Clinical Dataset Analysis & Feature Pruning Pipeline

This notebook executes an optimized clinical preprocessing pipeline on the **VitalDB Clinical Dataset (3,764 Patients)**, pruning unnecessary free-text diagnosis (`dx`), raw operation names (`opname`), and sparse equipment sizes down to **43 essential clinical features**.

### Workflow Steps:
1. **Feature Pruning**: Removes high-cardinality text (`dx`, `opname`) and sparse equipment sizes ($>75\%$ missing).
2. **Selected Essential Features**: Demographics, pre-op lab values, comorbidities, airway scores, intraoperative fluids, and vasopressors.
3. **Missing Value Imputation**: Median imputation for numerical variables and mode imputation for categorical features.
4. **Compact Categorical Encoding**: One-Hot Encoding for categorical features (`sex`, `department`, `optype`, `ane_type`, `approach`, `position`).
5. **Standardization & Scaling**: Z-score normalization (`StandardScaler`) for continuous numerical features.
6. **Export Machine-Learning Ready Dataset**: Saves clean output to `preprocessed_patient_metadata.csv`.

## 1. Environment Setup & Data Loading

In [ ]:
import os
import json
import numpy as np
import pandas as pd

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer

FIG_DIR = 'docs/metadata_eda_figures'
os.makedirs(FIG_DIR, exist_ok=True)

metadata_path = 'patient_metadata.csv'
df_raw = pd.read_csv(metadata_path)
print(f'[Loaded Raw Dataset] Patients: {len(df_raw)} | Raw Columns: {df_raw.shape[1]}')

## 2. Feature Selection & Pruning Unnecessary Columns

In [ ]:
# Define 43 Essential Clinical Parameters (Demographics, Pre-op Labs, Airway, Intraop Fluids, Outcomes)
essential_cols = [
    'caseid', 'subjectid', 'age', 'sex', 'height', 'weight', 'bmi', 'asa', 'emop',
    'department', 'optype', 'ane_type', 'approach', 'position',
    'preop_htn', 'preop_dm', 'preop_hb', 'preop_plt', 'preop_pt', 'preop_aptt',
    'preop_na', 'preop_k', 'preop_gluc', 'preop_alb', 'preop_ast', 'preop_alt', 'preop_bun', 'preop_cr',
    'cormack', 'airway', 'tubesize',
    'intraop_ebl', 'intraop_uo', 'intraop_crystalloid', 'intraop_colloid', 'intraop_rbc', 'intraop_ffp',
    'intraop_eph', 'intraop_phe', 'intraop_epi', 'intraop_ca',
    'icu_days', 'death_inhosp'
]

avail_cols = [c for c in essential_cols if c in df_raw.columns]
df_pruned = df_raw[avail_cols].copy()

dropped_cols = [c for c in df_raw.columns if c not in avail_cols]
print(f'✓ Successfully pruned dataset from {df_raw.shape[1]} columns down to {df_pruned.shape[1]} essential clinical columns.')
print(f'  • Dropped {len(dropped_cols)} unnecessary/sparse/text columns: {dropped_cols[:8]}...')

## 3. Missing Value Imputation (Median & Mode)

In [ ]:
num_cols = df_pruned.select_dtypes(include=[np.number]).columns.tolist()
cat_cols = df_pruned.select_dtypes(include=['object']).columns.tolist()

if 'caseid' in num_cols:
    num_cols.remove('caseid')
if 'subjectid' in num_cols:
    num_cols.remove('subjectid')

# 1. Median Imputation for Numerical Columns
num_imputer = SimpleImputer(strategy='median')
df_pruned[num_cols] = num_imputer.fit_transform(df_pruned[num_cols])
print(f'  • Median Imputed {len(num_cols)} numerical features.')

# 2. Mode Imputation for Categorical Columns
cat_imputer = SimpleImputer(strategy='most_frequent')
df_pruned[cat_cols] = cat_imputer.fit_transform(df_pruned[cat_cols])
print(f'  • Mode Imputed {len(cat_cols)} categorical features.')

## 4. Categorical Encoding & Z-Score Normalization

In [ ]:
# 1. One-Hot Encoding for Categorical Columns
df_processed = pd.get_dummies(df_pruned, columns=cat_cols, drop_first=True)
print(f'  • One-Hot Encoded categorical variables. Processed shape: {df_processed.shape}')

# 2. Z-Score Standardization for Continuous Numerical Columns
scaler = StandardScaler()
scaled_features = [c for c in num_cols if c not in ['death_inhosp', 'emop']]
df_processed[scaled_features] = scaler.fit_transform(df_processed[scaled_features])
print(f'  • Z-Score Normalized {len(scaled_features)} continuous numerical features.')

# 3. Save Clean Output CSV Files
out_csv = 'preprocessed_patient_metadata.csv'
docs_csv = 'docs/preprocessed_patient_metadata.csv'

df_processed.to_csv(out_csv, index=False)
df_processed.to_csv(docs_csv, index=False)

print('\n' + '=' * 75)
print(' CLEAN PREPROCESSING PIPELINE COMPLETE!')
print(f'  • Clean Output  : {out_csv} ({df_processed.shape[0]} rows x {df_processed.shape[1]} columns)')
print(f'  • Docs Copy    : {docs_csv}')
print('=' * 75)